# Sales Data Analysis Using Python
Analyze sales performance, monthly trends, top products, and city-wise revenue using Pandas, NumPy, and Matplotlib.

**Dataset note:** `data/sales_data.csv` is a small synthetic demo dataset for testing. Replace it with a real dataset for your final analysis.

## 1. Import libraries
If a package is missing in your environment, install it with `pip install -r requirements.txt`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Load the dataset
In Google Colab, you can upload your own CSV. If you skip upload, the notebook reads the included demo CSV. Expected fields: order date, product, quantity, unit price, and city.

In [ ]:
DATA_PATH = Path('data/sales_data.csv')
try:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
    if csv_files:
        DATA_PATH = Path(csv_files[0])
except ImportError:
    pass  # Not running in Google Colab; use the included demo file.

df = pd.read_csv(DATA_PATH)
print('Rows and columns:', df.shape)
display(df.head())

## 3. Inspect and clean the data
Normalize column names, parse dates, convert numeric fields, remove duplicate records, and handle missing values.

In [ ]:
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))

# Map common column-name variations to standard names.
aliases = {
    'orderdate': 'order_date', 'date': 'order_date', 'product_name': 'product',
    'item': 'product', 'qty': 'quantity', 'units': 'quantity',
    'price': 'unit_price', 'unitprice': 'unit_price', 'location': 'city'
}
df = df.rename(columns={c: aliases.get(c, c) for c in df.columns})
required = ['order_date', 'product', 'quantity', 'unit_price', 'city']
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}. Please check your CSV headers.')

df = df.drop_duplicates().copy()
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
for col in ['quantity', 'unit_price']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['product'] = df['product'].astype('string').str.strip()
df['city'] = df['city'].astype('string').str.strip()
df = df.dropna(subset=['order_date', 'product', 'quantity', 'unit_price', 'city'])
df = df[(df['quantity'] > 0) & (df['unit_price'] >= 0)].copy()
df['sales'] = df['quantity'] * df['unit_price']
df['month'] = df['order_date'].dt.to_period('M').astype(str)

print('Cleaned rows:', len(df))
display(df.head())
display(df.isna().sum().rename('missing_values').to_frame())

## 4. Key performance indicators
Calculate total revenue, units sold, order count (when an order ID exists), and average transaction value.

In [ ]:
total_revenue = df['sales'].sum()
units_sold = df['quantity'].sum()
order_count = df['order_id'].nunique() if 'order_id' in df.columns else len(df)
average_order_value = total_revenue / order_count if order_count else 0
print(f'Total revenue: {total_revenue:,.2f}')
print(f'Units sold: {units_sold:,.0f}')
print(f'Orders / records: {order_count:,}')
print(f'Average revenue per order / record: {average_order_value:,.2f}')

## 5. Monthly sales trend
Aggregate revenue by month and visualize the trend.

In [ ]:
monthly_sales = df.groupby('month', as_index=False)['sales'].sum()
display(monthly_sales)
plt.figure()
plt.plot(monthly_sales['month'], monthly_sales['sales'], marker='o')
plt.title('Monthly Sales Revenue')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Top-performing products
Rank products by total revenue.

In [ ]:
product_sales = (df.groupby('product', as_index=False)['sales'].sum()
                .sort_values('sales', ascending=False))
top_products = product_sales.head(10)
display(top_products)
plt.figure()
plt.bar(top_products['product'], top_products['sales'])
plt.title('Top 10 Products by Revenue')
plt.xlabel('Product')
plt.ylabel('Revenue')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. City-wise sales
Compare revenue across cities.

In [ ]:
city_sales = (df.groupby('city', as_index=False)['sales'].sum()
             .sort_values('sales', ascending=False))
display(city_sales)
plt.figure()
plt.bar(city_sales['city'], city_sales['sales'])
plt.title('Sales Revenue by City')
plt.xlabel('City')
plt.ylabel('Revenue')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Business insights
The statements below are calculated from whichever CSV you loaded.

In [ ]:
if not monthly_sales.empty and not top_products.empty and not city_sales.empty:
    best_month = monthly_sales.loc[monthly_sales['sales'].idxmax()]
    best_product = top_products.iloc[0]
    best_city = city_sales.iloc[0]
    print(f"Highest-revenue month: {best_month['month']} ({best_month['sales']:,.2f})")
    print(f"Top product by revenue: {best_product['product']} ({best_product['sales']:,.2f})")
    print(f"Top city by revenue: {best_city['city']} ({best_city['sales']:,.2f})")
else:
    print('Not enough valid data to generate insights.')

## 9. Save cleaned data and summaries
Export the cleaned dataset and analysis tables so they can be reused in a report.

In [ ]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
df.to_csv(output_dir / 'cleaned_sales_data.csv', index=False)
monthly_sales.to_csv(output_dir / 'monthly_sales.csv', index=False)
product_sales.to_csv(output_dir / 'product_sales.csv', index=False)
city_sales.to_csv(output_dir / 'city_sales.csv', index=False)
print('Analysis files saved in:', output_dir.resolve())